## 1. Workspace Components — Workspaces, Catalogs, Schemas


### Hierarchy

```
Account (Azure subscription)
 └── Metastore (one per region, top-level Unity Catalog container)
      ├── Workspace A ─┐  (workspaces are attached to a metastore and share its catalogs)
      ├── Workspace B ─┘
      └── Catalog            e.g. training, one_dbx_dev, samples
           └── Schema         e.g. bronze, silver, gold
                ├── Tables (managed / external)
                ├── Views / Materialized views
                ├── Volumes (non-tabular files: CSV, images, PDFs)
                ├── Functions (SQL / Python UDFs)
                └── Models (MLflow registered models)
```


In [0]:
%sql
-- Where am I? Metastore, default catalog/schema and identity of this session
SELECT current_metastore() AS metastore,
       current_catalog()   AS catalog,
       current_schema()    AS schema,
       current_user()      AS user;

## 2. Developer Interfaces — Notebooks, SQL Editor, Repos

| Interface | What it is | Best for | Key features |
|---|---|---|---|
| **Notebooks** | Multi-language, cell-based documents | Data engineering, exploration, ML, training material | Python / SQL / Markdown / shell cells via `%python`, `%sql`, `%md`, `%sh`; widgets; built-in visualisations; `_sqldf`; serverless or cluster compute; run as a job task |
| **SQL Editor** | Browser-based SQL IDE backed by SQL warehouses | BI analysts, ad-hoc SQL, saved queries, alerts | Schema browser, autocomplete, query parameters, query history, visualisations, feeds dashboards and alerts |
| **Repos (Git folders)** | Workspace folders linked to a remote Git repository (GitHub, Azure DevOps, GitLab, ...) | Version control, collaboration, CI/CD | Clone, branch, commit, push, pull, merge from the UI; jobs can run code directly from a Git ref |



## 3. Orchestration — Pipelines, Jobs

### Jobs vs Pipelines
Pipelines:Data transformations between tables, from bronze to silver, sivler to gold etc

Jobs:  Any task (notebooks, SQL, ML, pipelines, dashboards), A job often contains a **pipeline task**


##SQL Editor
Place to execute user sql

## Queries:

History for queries

## Dashbaord:

For creating new dashboard in Databricks

### Databricks Architecture — Control Plane, Compute Plane and Storage Plane

* Databricks runs on Azure, AWS and GCP.
* Some components are managed in the Databricks cloud account, others run in your own cloud subscription.
* Key idea: **Databricks manages the platform, your data stays in your Azure subscription.**
* Three planes: control plane, compute plane and storage plane.

#### Control Plane
* Hosted and managed by Databricks in its own Azure account, same region as the workspace.
* Runs the UI, REST APIs, job scheduler and cluster manager.
* Stores notebook / query / dashboard code, job definitions,  Catalog metadata, permissions, lineage, secrets, query history.

#### Compute Plane
* Where the data is processed: notebooks, SQL queries, jobs and pipelines run here.
* **Serverless**: runs in Databricks' Azure account; Databricks provisions, scales and patches it.
* **Classic**: cluster VMs run in your Azure subscription (managed resource group / VNet).
* Compute is temporary: reads from storage, processes in memory, writes back to storage; nothing is kept when it stops.

#### Storage Plane
* Lives in your Azure subscription (ADLS Gen2): Delta data files, `_delta_log` and volume files.
* Example: the `training` catalog stores data at `abfss://training@unityms.dfs.core.windows.net/` 

### Where information is stored

| Information | Stored in |
|---|---|
| **Table data** (Delta / Parquet files) | **Your cloud** – ADLS Gen2 | 
| **Volume files** (CSV, images, PDFs) | **Your cloud** – ADLS Gen2 | 
| **Delta transaction log** (`_delta_log`) | **Your cloud** |
| **Unity Catalog metadata** (catalogs, schemas, table definitions, grants, lineage) | **Databricks cloud** – control plane |
| **Notebook / query / dashboard source**, job definitions | **Databricks cloud** – control plane | 
| **Secrets** | **Databricks cloud** (Databricks-backed scope) or **your** Azure Key Vault (AKV-backed scope) | 
| **Query history, audit logs, billing** | **Databricks cloud** | 
| **Git folder content** | Remote Git provider + workspace copy in the control plane | 

## Compute
Compute is the **engine that runs your code**: the machines (CPU, memory) that do the work when you run a notebook, a query or a job.

---

### 1. Serverless Compute
* Databricks provisions and manages the infrastructure — **no cluster configuration needed**.
* Starts in **seconds**, auto-scales, and you pay only while code is running.
* Available for notebooks, jobs, pipelines and SQL.

### 2. All-Purpose Clusters (Interactive)
* Long-running clusters you **configure yourself** (VM size, runtime version, libraries, Spark config).
* Shared by multiple users for **interactive exploration**, notebook development and ad-hoc analysis.
* You control auto-scaling limits, idle-termination timeout and init scripts.

### 3. Job Clusters
* **Ephemeral clusters** created automatically when a job starts and terminated when it finishes.
* Optimised for **production workloads** — lower cost because they don't sit idle.
* Each job task can have its own cluster configuration.

### 4. SQL Warehouses
* Compute dedicated to **SQL queries, dashboards, alerts and BI tool connections** (e.g. Power BI, Tableau).
* Two flavours:
  - **Serverless SQL warehouse** — Databricks-managed, fastest start-up.
  - **Pro / Classic SQL warehouse** — runs in your Azure subscription, more configuration options.
* Auto-scales based on query concurrency; auto-stops after idle period.

### 5. GPU Clusters
* Clusters with **GPU-enabled VMs** (e.g. NVIDIA A10, T4, H100) for deep learning, LLM fine-tuning and inference.
* Available as all-purpose or job clusters with ML Runtime pre-installed (PyTorch, TensorFlow, etc.).
* Also available as **Serverless GPU** via AI Runtime for managed GPU workloads.

---

| Type | Managed by | Best for | Start-up | Cost model |
|---|---|---|---|---|
| Serverless | Databricks | Notebooks, jobs, pipelines, SQL | Seconds | Pay per use |
| All-Purpose Cluster | You | Interactive dev, exploration | Minutes | Running time |
| Job Cluster | You (auto-created) | Scheduled production jobs | Minutes | Job duration only |
| SQL Warehouse | Databricks / You | SQL, dashboards, BI | Seconds–minutes | Query time / running time |
| GPU Cluster | You / Databricks | ML training, deep learning, LLMs | Minutes | Running time (GPU VMs) |



##Decison Matrix for Classic/Serverless
<img src="/Volumes/training/bronze/images/Screenshot 2026-10-19 at 17.14.34.png" alt="Decision Matrix for Classic/Serverless" width="1000"/>
